# Sprint 12. Estabilidad de ResNet50 entre semillas

Comparación controlada de dos estrategias de fine-tuning: `conv5` completo frente a `conv5_block3`.

Se reutilizan los resultados de semilla 42 de los Sprints 09 y 11, sin volver a entrenarlos. Se entrenan cuatro corridas nuevas: semilla 123 y 2026 para cada estrategia.

La comparación usa únicamente las 127 imágenes de `validation_original`. Las 43 imágenes `260423` y TEST no se utilizan para ajustar configuraciones ni se evalúan en este sprint. Las conclusiones son descriptivas: la validación ya se utilizó en experimentos anteriores.

## Uso

Usar el kernel `palta_gpu (CUDA)` en Ubuntu WSL2. Ejecutar las secciones 1 a 4 en orden. En la sección 5 activar una sola corrida, esperar a que termine, poner `EJECUTAR_EXPERIMENTO = False` y guardar. Para cada nueva corrida, cambiar modo y semilla; las carpetas de resultados no se sobrescriben.

Orden sugerido: `conv5/s123`, `conv5_block3/s123`, `conv5/s2026`, `conv5_block3/s2026`. Consultar resultados en la sección 6. **No usar Run All.**

## 1. Rutas y configuración

In [1]:
from pathlib import Path, PureWindowsPath
from collections import Counter
import hashlib
import importlib.metadata
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "README.md").is_file()
             and (p / "utilidades_imagenes.py").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("Abra el notebook dentro de PROYECTO_TI_2 en VS Code/WSL.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from utilidades_imagenes import encontrar_raiz, resolver_dataset, cargar_rgb, ajustar_tamano

RAIZ = encontrar_raiz()
DATASET = resolver_dataset(RAIZ, "../DATASET_PALTA")
MANIFIESTO_RUTA = RAIZ / "7. resultados/validacion_por_fecha/manifiesto_260423.csv"
EDA = RAIZ / "7. resultados/eda"
AUDITORIA_ORIGEN = RAIZ / "7. resultados/entrenamiento/FT_D1_PHOTO_s42_01/auditoria.json"
SALIDA = RAIZ / "7. resultados/estabilidad_semillas"
SEMILLA = 123  # valor predeterminado de funciones; cada corrida lo recibe explícitamente
TAMANO = (256, 512)  # alto, ancho
TAMANO_ENTRADA = TAMANO
CLASES = ["PICADURA", "RUSSET"]
SEMILLAS_NUEVAS = (123, 2026)
MODOS = ("conv5", "conv5_block3")

# Se define antes de importar TensorFlow.
os.environ.setdefault("KERAS_HOME", str(RAIZ / ".cache" / "keras"))

print("Proyecto:", RAIZ)
print("Dataset:", DATASET)
print("Manifiesto:", MANIFIESTO_RUTA.relative_to(RAIZ))
print("Corridas pendientes:", [f"{m}/s{s}" for s in SEMILLAS_NUEVAS for m in MODOS])
print("Python:", sys.executable)

print("Comparación: conv5 frente a conv5_block3, Adam, mismas particiones.")
print("Semilla 42: usar resultados existentes, sin reentrenar.")
print("No se calcularán predicciones de 260423 ni TEST.")


Proyecto: /mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/PROYECTO_TI_2
Dataset: /mnt/c/Users/TCI_AUTO/Documents/Maestría UNI/4_2. Proyecto de Invetigación  2/DATASET_PALTA
Manifiesto: 7. resultados/validacion_por_fecha/manifiesto_260423.csv
Corridas pendientes: ['conv5/s123', 'conv5_block3/s123', 'conv5/s2026', 'conv5_block3/s2026']
Python: /home/tci_auto/.venvs/palta_gpu/bin/python
Comparación: conv5 frente a conv5_block3, Adam, mismas particiones.
Semilla 42: usar resultados existentes, sin reentrenar.
No se calcularán predicciones de 260423 ni TEST.


## 2. Auditoría y particiones fijas

In [2]:
def hash_archivo(ruta):
    calculo = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            calculo.update(bloque)
    return calculo.hexdigest()


def firma_tabla(tabla):
    texto = "\n".join(tabla["archivo"] + ":" + tabla["sha256"])
    return hashlib.sha256(texto.encode()).hexdigest()


def leer_auditoria():
    carpeta = RAIZ / "7. resultados" / "eda"
    resumen = json.loads((carpeta / "eda_resumen.json").read_text(encoding="utf-8-sig"))
    inventario = pd.read_csv(carpeta / "eda_inventario.csv", encoding="utf-8-sig", keep_default_na=False)
    duplicados = pd.read_csv(carpeta / "eda_duplicados.csv", encoding="utf-8-sig", keep_default_na=False)
    columnas = {"archivo", "sha256", "sha_pixeles", "particion", "clase", "imagen", "error"}
    if not columnas.issubset(inventario.columns):
        raise ValueError("El inventario no contiene la auditoria necesaria.")
    if inventario.empty or inventario["archivo"].duplicated().any():
        raise ValueError("El inventario esta vacio o contiene rutas repetidas.")
    if len(inventario) != resumen["archivos"] or len(inventario) != resumen["imagenes"]:
        raise ValueError("Los conteos no coinciden. Revise el EDA.")
    if resumen["corruptas"] or inventario["error"].ne("").any():
        raise ValueError("El EDA registra errores de lectura.")
    if not inventario["imagen"].astype(str).str.lower().isin(["true", "1"]).all():
        raise ValueError("El inventario incluye archivos no reconocidos como imagenes.")
    if firma_tabla(inventario) != resumen["sha256_dataset"]:
        raise ValueError("La firma del inventario no coincide con el resumen.")
    for campo in ("sha256", "sha_pixeles"):
        if not inventario[campo].astype(str).str.fullmatch(r"[0-9a-f]{64}").all():
            raise ValueError(f"Hashes incompletos en {campo}.")
    for registro in inventario.itertuples(index=False):
        ruta = Path(registro.archivo)
        win = PureWindowsPath(registro.archivo)
        if (ruta.is_absolute() or win.drive or win.root or ".." in ruta.parts
                or "\\" in registro.archivo or len(ruta.parts) < 3):
            raise ValueError("El inventario debe usar rutas relativas dentro del dataset.")
        if ruta.parts[:2] != (registro.particion, registro.clase):
            raise ValueError(f"Ruta y etiqueta incoherentes: {registro.archivo}")
        if registro.particion not in {"train", "validation", "test"}:
            raise ValueError("Particion no reconocida.")
        if registro.clase not in {"PICADURA", "RUSSET"}:
            raise ValueError("Clase no reconocida.")

    requeridas = {"tipo", "archivo", "archivo_coincidente", "particion",
                  "particion_coincidente", "clase", "clase_coincidente"}
    if not requeridas.issubset(duplicados.columns):
        raise ValueError("El reporte de duplicados esta incompleto.")
    exactos = duplicados[duplicados["tipo"].isin(["bytes_exactos", "pixeles_exactos"])]
    indexado = inventario.set_index("archivo")
    pares_fuga, pares_conflicto, excluidos = set(), set(), set()
    for fila in exactos.itertuples(index=False):
        a, b = fila.archivo, fila.archivo_coincidente
        if a == b or a not in indexado.index or b not in indexado.index:
            raise ValueError("El reporte de duplicados contiene una ruta invalida.")
        ra, rb = indexado.loc[a], indexado.loc[b]
        if (ra.particion, rb.particion, ra.clase, rb.clase) != (
                fila.particion, fila.particion_coincidente, fila.clase, fila.clase_coincidente):
            raise ValueError("Duplicados e inventario no coinciden.")
        clave = "sha256" if fila.tipo == "bytes_exactos" else "sha_pixeles"
        if ra[clave] != rb[clave]:
            raise ValueError("El par reportado no coincide con los hashes del inventario.")
        par = tuple(sorted([a, b]))
        fuga, conflicto = ra.particion != rb.particion, ra.clase != rb.clase
        if fuga:
            pares_fuga.add(par)
        if conflicto:
            pares_conflicto.add(par)
        if fuga or conflicto:
            excluidos.update([a, b])
    if (len(pares_fuga) != resumen["pares_exactos_entre_particiones"]
            or len(pares_conflicto) != resumen["pares_exactos_etiquetas_distintas"]):
        raise ValueError("Los conteos de pares no coinciden con el resumen.")

    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].copy()
    for clave in ("sha256", "sha_pixeles"):
        grupos = elegibles.groupby(clave)[["particion", "clase"]].nunique()
        if (grupos > 1).any().any():
            raise ValueError("Persisten duplicados entre particiones o clases. Revise el EDA.")
    return resumen, inventario, sorted(excluidos)


def verificar_auditoria(dataset, aprobar_exclusiones=False):
    resumen, inventario, excluidos = leer_auditoria()
    dataset = Path(dataset).resolve()
    rutas = set(inventario["archivo"])
    locales = {p.relative_to(dataset).as_posix() for p in dataset.rglob("*") if p.is_file()}
    if locales != rutas:
        raise RuntimeError("El contenido del dataset cambio. Revise las rutas y actualice el EDA.")
    total = len(inventario)
    for numero, registro in enumerate(inventario.itertuples(index=False), 1):
        ruta = (dataset / registro.archivo).resolve()
        if not ruta.is_relative_to(dataset):
            raise ValueError("Un enlace apunta fuera del dataset.")
        if hash_archivo(ruta) != registro.sha256:
            raise RuntimeError(f"Archivo distinto al auditado: {registro.archivo}")
        if numero % 100 == 0 or numero == total:
            print(f"Integridad: {numero}/{total}")
    if excluidos and not aprobar_exclusiones:
        raise RuntimeError(
            "Integridad correcta; deben aprobarse las exclusiones. "
            "No se entreno ni se modificaron imagenes."
        )
    elegibles = inventario.loc[~inventario["archivo"].isin(excluidos)].sort_values("archivo")
    return {
        "sha256_dataset": resumen["sha256_dataset"],
        "sha256_manifiesto": firma_tabla(elegibles),
        "exclusiones": excluidos,
        "archivos_originales": int(len(inventario)),
        "archivos_elegibles": int(len(elegibles)),
        "alcance": "Integridad exacta; independencia por fruto/lote pendiente de trazabilidad.",
    }

# La auditoría es la misma del entrenamiento original; la división por fecha es nueva.
for requisito in (MANIFIESTO_RUTA, AUDITORIA_ORIGEN,
                  EDA / "eda_inventario.csv", EDA / "eda_duplicados.csv",
                  EDA / "eda_resumen.json"):
    if not requisito.is_file():
        raise FileNotFoundError(f"Archivo necesario no encontrado: {requisito}")

referencia = json.loads(AUDITORIA_ORIGEN.read_text(encoding="utf-8"))
resumen_eda, inventario_eda, excluidos_eda = leer_auditoria()
if sorted(excluidos_eda) != sorted(referencia["exclusiones"]):
    raise ValueError("Las cuatro exclusiones no coinciden con el entrenamiento histórico.")
if (resumen_eda["sha256_dataset"] != referencia["sha256_dataset"] or
        firma_tabla(inventario_eda.loc[
            ~inventario_eda["archivo"].isin(excluidos_eda)
        ].sort_values("archivo")) != referencia["sha256_manifiesto"]):
    raise ValueError("La auditoría EDA no coincide con el protocolo del entrenamiento original.")

manifiesto = pd.read_csv(MANIFIESTO_RUTA, encoding="utf-8-sig", keep_default_na=False)
columnas = {"archivo", "clase", "conjunto"}
if not columnas.issubset(manifiesto.columns) or manifiesto.empty:
    raise ValueError("El manifiesto necesita archivo, clase y conjunto.")
if manifiesto["archivo"].duplicated().any():
    raise ValueError("El manifiesto contiene rutas repetidas.")

esperado = inventario_eda.loc[
    ~inventario_eda["archivo"].isin(excluidos_eda)
    & inventario_eda["particion"].isin(["train", "validation"]),
    ["archivo", "clase", "particion", "sha256", "sha_pixeles"],
].copy()
esperado["conjunto"] = np.where(
    esperado["particion"].eq("validation"), "validation_original",
    np.where(esperado["archivo"].str.contains(r"-260423-", regex=True),
             "evaluacion_exploratoria_260423", "train_revisado"),
)

comparar = ["archivo", "clase", "conjunto"]
a = manifiesto[comparar].sort_values(comparar).reset_index(drop=True)
b = esperado[comparar].sort_values(comparar).reset_index(drop=True)
pd.testing.assert_frame_equal(a, b, check_dtype=False)

conteos = Counter(zip(manifiesto["conjunto"], manifiesto["clase"]))
conteos_esperados = {
    ("train_revisado", "PICADURA"): 474,
    ("train_revisado", "RUSSET"): 55,
    ("validation_original", "PICADURA"): 106,
    ("validation_original", "RUSSET"): 21,
    ("evaluacion_exploratoria_260423", "PICADURA"): 21,
    ("evaluacion_exploratoria_260423", "RUSSET"): 22,
}
if conteos != conteos_esperados or len(manifiesto) != 699:
    raise ValueError(f"Conteos no esperados: {dict(conteos)}")

# El EDA anterior permitía duplicados dentro de TRAIN. Al mover 43 TIFF a
# evaluación, debemos comprobar de nuevo los duplicados cruzados por bytes/píxeles.
con_hash = manifiesto.merge(
    esperado[["archivo", "sha256", "sha_pixeles"]],
    on="archivo", how="left", validate="one_to_one",
)
for clave in ("sha256", "sha_pixeles"):
    conflicto = con_hash.groupby(clave)["conjunto"].nunique()
    conflicto = conflicto[conflicto > 1]
    if not conflicto.empty:
        afectados = con_hash.loc[con_hash[clave].isin(conflicto.index),
                                ["archivo", "clase", "conjunto", clave]]
        print(afectados.to_string(index=False))
        raise RuntimeError(f"Hay duplicados exactos entre conjuntos según {clave}. No entrenar.")

for archivo in manifiesto["archivo"]:
    ruta = Path(archivo)
    win = PureWindowsPath(archivo)
    if ruta.is_absolute() or win.drive or win.root or ".." in ruta.parts or "\\" in archivo:
        raise ValueError(f"Ruta fuera de dataset: {archivo}")
    destino = (DATASET / archivo).resolve()
    if not destino.is_relative_to(DATASET.resolve()) or not destino.is_file():
        raise FileNotFoundError(f"Imagen no localizada dentro del dataset: {archivo}")

CONJUNTOS = {}
for nombre in ("train_revisado", "validation_original", "evaluacion_exploratoria_260423"):
    tabla = manifiesto.loc[manifiesto["conjunto"] == nombre, ["archivo", "clase"]].copy()
    tabla = tabla.sort_values("archivo").reset_index(drop=True)
    tabla["particion"] = tabla["archivo"].str.split("/").str[0]
    tabla["etiqueta"] = tabla["clase"].map({"PICADURA": 0, "RUSSET": 1}).astype("int32")
    CONJUNTOS[nombre] = tabla

print("Auditoría coherente. No hay duplicados exactos cruzados en la nueva división.")
display(pd.crosstab(manifiesto["conjunto"], manifiesto["clase"]))
print("Total TRAIN + VALIDATION + evaluación exploratoria:", len(manifiesto))
print("No se entrenó ningún modelo ni se calcularon predicciones de TEST.")


Auditoría coherente. No hay duplicados exactos cruzados en la nueva división.


clase,PICADURA,RUSSET
conjunto,,
evaluacion_exploratoria_260423,21,22
train_revisado,474,55
validation_original,106,21


Total TRAIN + VALIDATION + evaluación exploratoria: 699
No se entrenó ningún modelo ni se calcularon predicciones de TEST.


## 3. Balanceo D1 y preprocesamiento

In [3]:
def plan_copias_russet(tabla_train, copias, semilla=SEMILLA):
    """Registra copias de los frutos RUSSET sin alterar el inventario."""
    if not isinstance(copias, int) or copias < 0 or copias > 3:
        raise ValueError("Use entre 0 y 3 copias por fruto.")
    russet = tabla_train.loc[tabla_train["clase"] == "RUSSET", "archivo"].tolist()
    filas = []
    for archivo in russet:
        for numero in range(1, copias + 1):
            clave = f"{semilla}|{archivo}|{numero}".encode("utf-8")
            entero = int.from_bytes(hashlib.sha256(clave).digest()[:8], "big")
            rng = np.random.default_rng(entero)
            filas.append({"archivo": archivo, "copia": numero, "etiqueta": 1,
                          "brillo": float(rng.uniform(0.92, 1.08)),
                          "contraste": float(rng.uniform(0.90, 1.10))})
    return pd.DataFrame(filas, columns=["archivo", "copia", "etiqueta", "brillo", "contraste"])


def aplicar_cambio_fotometrico(matriz, mascara, brillo, contraste):
    """Cambia la intensidad y el contraste sin mover pixeles ni modificar el relleno."""
    matriz = np.asarray(matriz, dtype=np.float32)
    mascara = np.asarray(mascara, dtype=bool)
    if matriz.ndim != 3 or matriz.shape[-1] != 3 or mascara.shape != matriz.shape[:2]:
        raise ValueError("Revise las dimensiones RGB y la mascara.")
    if not (0.85 <= brillo <= 1.15 and 0.85 <= contraste <= 1.15):
        raise ValueError("El cambio solicitado esta fuera del rango previsto.")
    resultado = matriz.copy()
    if mascara.any():
        pixeles = matriz[mascara]
        media = pixeles.mean(axis=0)
        resultado[mascara] = np.clip((pixeles - media) * contraste * brillo + media * brillo, 0, 255)
    return resultado




def pesos_para_etiquetas(etiquetas, clases):
    """Calcula class_weight con la distribucion efectiva de cada condicion."""
    from sklearn.utils.class_weight import compute_class_weight
    etiquetas = np.asarray(etiquetas, dtype=np.int32)
    valores = np.arange(len(clases), dtype=np.int32)
    if set(np.unique(etiquetas)) != set(valores):
        raise ValueError("El entrenamiento debe conservar ambas clases.")
    pesos = compute_class_weight("balanced", classes=valores, y=etiquetas)
    return {int(k): float(v) for k, v in zip(valores, pesos)}

BRILLO_EXTRA = (0.95, 1.05)
CONTRASTE_EXTRA = (0.95, 1.05)


def construir_manifiesto_entrenamiento(tabla_train, plan):
    base = tabla_train[["archivo", "etiqueta"]].copy()
    base["copia"] = 0
    base["brillo_d1"] = 1.0
    base["contraste_d1"] = 1.0
    extras = plan.rename(columns={"brillo": "brillo_d1", "contraste": "contraste_d1"})[
        ["archivo", "etiqueta", "copia", "brillo_d1", "contraste_d1"]
    ].copy()
    manifiesto = pd.concat([base, extras], ignore_index=True)
    if manifiesto["archivo"].isna().any():
        raise ValueError("El manifiesto incluye rutas vacias.")
    if not set(manifiesto["etiqueta"]) == {0, 1}:
        raise ValueError("El entrenamiento debe incluir PICADURA y RUSSET.")
    return manifiesto


def factores_para_visita(indice_global, archivo, semilla=SEMILLA):
    clave = f"{semilla}|{int(indice_global)}|{archivo}".encode("utf-8")
    entero = int.from_bytes(hashlib.sha256(clave).digest()[:8], "big")
    generador = np.random.default_rng(entero)
    brillo = float(generador.uniform(*BRILLO_EXTRA))
    contraste = float(generador.uniform(*CONTRASTE_EXTRA))
    return brillo, contraste


def crear_train_fotometrico(manifiesto, dataset, tamano=TAMANO, lote=1, semilla=SEMILLA, foto_extra=True):
    """Usa D1 en ambas etapas; ±5 % extra solo cuando foto_extra=True."""
    import tensorflow as tf
    if lote != 1:
        raise ValueError("Este protocolo utiliza lote de tamaño 1.")
    tamano = tuple(map(int, tamano))
    dataset = Path(dataset)
    if manifiesto.empty:
        raise ValueError("El entrenamiento esta vacio.")

    archivos = manifiesto["archivo"].astype(str).tolist()
    etiquetas = manifiesto["etiqueta"].to_numpy(dtype=np.int32)
    copias = manifiesto["copia"].to_numpy(dtype=np.int32)
    brillos = manifiesto["brillo_d1"].to_numpy(dtype=np.float32)
    contrastes = manifiesto["contraste_d1"].to_numpy(dtype=np.float32)

    secuencia = tf.data.Dataset.from_tensor_slices(
        (archivos, etiquetas, copias, brillos, contrastes)
    )
    secuencia = secuencia.shuffle(
        buffer_size=len(manifiesto), seed=semilla, reshuffle_each_iteration=True
    ).repeat().enumerate()

    def cargar_y_transformar(indice, ruta_relativa, etiqueta, copia, brillo_d1, contraste_d1):
        ruta_bytes = ruta_relativa.item() if hasattr(ruta_relativa, "item") else ruta_relativa
        ruta = ruta_bytes.decode("utf-8") if isinstance(ruta_bytes, bytes) else str(ruta_bytes)
        imagen = cargar_rgb(dataset / ruta)
        rgb, mascara = ajustar_tamano(imagen, tamano)
        rgb = np.asarray(rgb, dtype=np.float32)
        mascara = np.asarray(mascara, dtype=bool)
        if int(copia) > 0:
            rgb = aplicar_cambio_fotometrico(
                rgb, mascara, float(brillo_d1), float(contraste_d1)
            )
        if foto_extra:
            brillo, contraste = factores_para_visita(int(indice), ruta, semilla)
            rgb = aplicar_cambio_fotometrico(rgb, mascara, brillo, contraste)
        if not np.isfinite(rgb).all():
            raise ValueError(f"La imagen transformada contiene valores invalidos: {ruta}")
        return rgb.astype(np.float32), np.int32(etiqueta)

    def mapear(indice, registro):
        ruta, etiqueta, copia, brillo_d1, contraste_d1 = registro
        imagen, clase = tf.numpy_function(
            cargar_y_transformar,
            [indice, ruta, etiqueta, copia, brillo_d1, contraste_d1],
            Tout=[tf.float32, tf.int32]
        )
        imagen.set_shape((*tamano, 3))
        clase.set_shape(())
        return imagen, clase

    opciones = tf.data.Options()
    opciones.threading.private_threadpool_size = 2
    opciones.experimental_deterministic = True
    secuencia = secuencia.with_options(opciones)
    return secuencia.map(mapear, num_parallel_calls=1).batch(lote).prefetch(1)

# Cada semilla modifica de forma controlada el muestreo, las transformaciones
# y la inicialización. Dentro de una misma semilla, ambos modos usan el mismo D1.
for semilla in SEMILLAS_NUEVAS:
    plan = plan_copias_russet(CONJUNTOS["train_revisado"], 1, semilla=semilla)
    efectivo = construir_manifiesto_entrenamiento(CONJUNTOS["train_revisado"], plan)
    pesos = pesos_para_etiquetas(efectivo["etiqueta"].to_numpy(), CLASES)
    if len(plan) != 55 or len(efectivo) != 584:
        raise ValueError("El plan D1 debe producir 55 copias y 584 muestras.")
    print(f"Semilla {semilla}: TRAIN=529, D1=55, efectivo=584, class_weight={pesos}")
print("No se ha entrenado ningún modelo.")


Semilla 123: TRAIN=529, D1=55, efectivo=584, class_weight={0: 0.6160337552742616, 1: 2.6545454545454548}
Semilla 2026: TRAIN=529, D1=55, efectivo=584, class_weight={0: 0.6160337552742616, 1: 2.6545454545454548}
No se ha entrenado ningún modelo.


## 4. ResNet50 y entrenamiento por corrida

In [4]:
def construir_modelo(numero_clases, tamano=TAMANO_ENTRADA,
                      volteo_horizontal=False, semilla=SEMILLA, pesos="imagenet"):
    import tensorflow as tf
    from tensorflow.keras.applications.resnet50 import preprocess_input

    if numero_clases < 2:
        raise ValueError("El clasificador necesita al menos dos clases.")
    tf.keras.utils.set_random_seed(semilla)
    entrada = tf.keras.Input(shape=(*tamano, 3), dtype="float32", name="imagen_rgb")
    x = entrada
    if volteo_horizontal:
        x = tf.keras.layers.RandomFlip("horizontal", seed=semilla, name="aumento_train")(x)
    x = tf.keras.layers.Lambda(preprocess_input, name="preprocesamiento_resnet50")(x)
    modelo_base = tf.keras.applications.ResNet50(
        input_shape=(*tamano, 3), include_top=False, weights=pesos, pooling="avg",
    )
    modelo_base.trainable = False
    x = modelo_base(x, training=False)
    x = tf.keras.layers.Dropout(0.35, seed=semilla)(x)
    salida = tf.keras.layers.Dense(
        numero_clases, activation="softmax", name="clase",
        kernel_initializer=tf.keras.initializers.GlorotUniform(seed=semilla),
    )(x)
    modelo = tf.keras.Model(entrada, salida, name="resnet50_palta")
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy", metrics=["accuracy"],
    )
    return modelo


def crear_dataset_evaluacion(tabla):
    """Lectura RGB 0-255 en el orden de tabla, sin aleatoriedad ni aumentos."""
    import tensorflow as tf
    rutas = tabla["archivo"].astype(str).tolist()
    etiquetas = tabla["etiqueta"].to_numpy(dtype=np.int32)
    secuencia = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))

    def leer(ruta_relativa):
        valor = ruta_relativa.item() if hasattr(ruta_relativa, "item") else ruta_relativa
        ruta = valor.decode("utf-8") if isinstance(valor, bytes) else str(valor)
        rgb, _ = ajustar_tamano(cargar_rgb(DATASET / ruta), TAMANO)
        rgb = np.asarray(rgb, dtype=np.float32)
        if rgb.shape != (*TAMANO, 3) or not np.isfinite(rgb).all():
            raise ValueError(f"Imagen inválida en evaluación: {ruta}")
        return rgb

    def mapear(ruta, etiqueta):
        imagen = tf.numpy_function(leer, [ruta], tf.float32)
        imagen.set_shape((*TAMANO, 3))
        etiqueta.set_shape(())
        return imagen, etiqueta

    opciones = tf.data.Options()
    opciones.experimental_deterministic = True
    opciones.threading.private_threadpool_size = 2
    return secuencia.with_options(opciones).map(mapear, num_parallel_calls=1).batch(1).prefetch(1)


def callbacks_etapa(tf, carpeta, etapa):
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                         restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(
            str(carpeta / f"mejor_{etapa}.weights.h5"), monitor="val_loss",
            save_best_only=True, save_weights_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2),
        tf.keras.callbacks.CSVLogger(str(carpeta / f"historial_{etapa}.csv")),
        tf.keras.callbacks.TerminateOnNaN(),
    ]


def evaluar_guardar(modelo, tabla, nombre, carpeta, firma_protocolo):
    """Graba métricas y predicciones una vez, después de fijar el checkpoint."""
    import tensorflow as tf
    import matplotlib.pyplot as plt
    from sklearn.metrics import (
        accuracy_score, f1_score, recall_score, precision_score,
        balanced_accuracy_score, classification_report,
        confusion_matrix, ConfusionMatrixDisplay,
        average_precision_score, roc_auc_score, brier_score_loss,
    )
    datos = tabla.reset_index(drop=True)
    secuencia = crear_dataset_evaluacion(datos)
    probabilidades = np.asarray(modelo.predict(secuencia, verbose=0))
    if probabilidades.shape != (len(datos), 2) or not np.isfinite(probabilidades).all():
        raise RuntimeError(f"Predicciones inválidas: {nombre}")
    if not np.allclose(probabilidades.sum(axis=1), 1.0, atol=1e-4):
        raise RuntimeError("Las probabilidades no suman 1.")
    reales = datos["etiqueta"].to_numpy(dtype=np.int32)
    estimadas = np.argmax(probabilidades, axis=1)
    prusset = probabilidades[:, 1]
    informe = classification_report(
        reales, estimadas, labels=[0, 1], target_names=CLASES,
        output_dict=True, zero_division=0,
    )
    metricas = {
        "estado": "completo", "particion": nombre, "n_imagenes": int(len(datos)),
        "firma_protocolo": firma_protocolo,
        "accuracy": float(accuracy_score(reales, estimadas)),
        "macro_f1": float(f1_score(reales, estimadas, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(reales, estimadas)),
        "recall_russet": float(recall_score(reales, estimadas, pos_label=1, zero_division=0)),
        "precision_russet": float(precision_score(reales, estimadas, pos_label=1, zero_division=0)),
        "average_precision_russet": float(average_precision_score(reales, prusset)),
        "roc_auc_russet": float(roc_auc_score(reales, prusset)),
        "brier_russet": float(brier_score_loss(reales, prusset)),
    }
    (carpeta / f"metricas_{nombre}.json").write_text(
        json.dumps(metricas, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    resultados = datos[["archivo", "clase", "etiqueta"]].copy()
    resultados["prediccion"] = [CLASES[int(i)] for i in estimadas]
    resultados["probabilidad_PICADURA"] = probabilidades[:, 0]
    resultados["probabilidad_RUSSET"] = prusset
    resultados.to_csv(carpeta / f"predicciones_{nombre}.csv", index=False)
    pd.DataFrame(informe).transpose().to_csv(carpeta / f"metricas_por_clase_{nombre}.csv")
    matriz = confusion_matrix(reales, estimadas, labels=[0, 1])
    pd.DataFrame(matriz, index=CLASES, columns=CLASES).to_csv(
        carpeta / f"matriz_confusion_{nombre}.csv"
    )
    import matplotlib.pyplot as plt
    vista = ConfusionMatrixDisplay(matriz, display_labels=CLASES).plot(values_format="d")
    vista.ax_.set_title(f"ResNet50 - {nombre}")
    vista.figure_.tight_layout()
    vista.figure_.savefig(carpeta / f"matriz_confusion_{nombre}.png", dpi=150)
    plt.close(vista.figure_)
    print(f"{nombre}: Macro-F1={metricas['macro_f1']:.4f} | "
          f"Recall RUSSET={metricas['recall_russet']:.4f} | "
          f"Accuracy={metricas['accuracy']:.4f}")
    return metricas

def ejecutar_una_corrida(modo, semilla):
    """Una única corrida nueva por modo/semilla. No evalúa 260423 ni TEST."""
    import tensorflow as tf
    import matplotlib.pyplot as plt

    modo = str(modo)
    semilla = int(semilla)
    if modo not in MODOS or semilla not in SEMILLAS_NUEVAS:
        raise ValueError("Solo se permiten conv5/conv5_block3 con semillas 123 o 2026.")

    etiqueta = "CONV5" if modo == "conv5" else "CONV5B3"
    nombre = f"RN50_{etiqueta}_s{semilla}_01"
    destino = SALIDA / nombre
    if destino.exists():
        raise FileExistsError(f"Esta corrida ya tiene una carpeta; no sobrescribir: {destino}")

    if not DATASET.is_dir():
        raise FileNotFoundError(f"No se encontró DATASET: {DATASET}")
    gpus = tf.config.list_physical_devices("GPU")
    if not gpus:
        raise RuntimeError("No se detectó GPU. Seleccione palta_gpu (CUDA) en WSL2.")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError:
            pass

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(semilla)
    try:
        tf.config.experimental.enable_op_determinism()
    except (AttributeError, RuntimeError):
        pass

    auditoria_actual = verificar_auditoria(DATASET, aprobar_exclusiones=True)
    for campo in ("sha256_dataset", "sha256_manifiesto"):
        if auditoria_actual[campo] != referencia[campo]:
            raise RuntimeError(f"La identidad auditada del dataset cambió: {campo}")
    if sorted(auditoria_actual["exclusiones"]) != sorted(referencia["exclusiones"]):
        raise RuntimeError("Las cuatro exclusiones difieren del protocolo original.")

    plan_d1 = plan_copias_russet(CONJUNTOS["train_revisado"], 1, semilla=semilla)
    train_efectivo = construir_manifiesto_entrenamiento(CONJUNTOS["train_revisado"], plan_d1)
    pesos_efectivos = pesos_para_etiquetas(train_efectivo["etiqueta"].to_numpy(), CLASES)
    if len(train_efectivo) != 584 or len(plan_d1) != 55:
        raise ValueError("El entrenamiento debe mantener D1: 584 muestras.")

    # No cargar cabezas ni pesos de los entrenamientos históricos.
    modelo = construir_modelo(len(CLASES), TAMANO, False, semilla, pesos="imagenet")
    base = next(c for c in modelo.layers if isinstance(c, tf.keras.Model))
    if base.trainable or any(c.trainable for c in base.layers):
        raise RuntimeError("La primera etapa debe congelar ResNet50 por completo.")
    validacion = crear_dataset_evaluacion(CONJUNTOS["validation_original"])
    entrenamiento_cabeza = crear_train_fotometrico(
        train_efectivo, DATASET, TAMANO, lote=1, semilla=semilla, foto_extra=False
    )
    entrenamiento_ft = crear_train_fotometrico(
        train_efectivo, DATASET, TAMANO, lote=1, semilla=semilla, foto_extra=True
    )

    configuracion = {
        "sprint": 12,
        "nombre_ejecucion": nombre,
        "modo_finetune": modo,
        "semilla": semilla,
        "arquitectura": "ResNet50 ImageNet + Dropout(0.35) + Dense(2)",
        "optimizador_cabeza": "Adam",
        "optimizador_finetune": "Adam",
        "origen_pesos_backbone": "ImageNet",
        "carga_cabeza_historica": False,
        "entrenamiento_independiente": True,
        "clases": CLASES,
        "tamano_entrada": list(TAMANO),
        "train": 529,
        "validation": 127,
        "evaluacion_exploratoria_260423": 43,
        "evaluacion_exploratoria_consultada": False,
        "test_consultado": False,
        "train_efectivo_d1": len(train_efectivo),
        "copias_russet": len(plan_d1),
        "pesos_clase": pesos_efectivos,
        "epocas_max_cabeza": 8,
        "epocas_max_finetune": 8,
        "learning_rate_cabeza": 1e-3,
        "learning_rate_finetune": 1e-5,
        "dropout": 0.35,
        "lote": 1,
        "batch_normalization": "congelada",
        "fotometria_d1": {"brillo": [0.92, 1.08], "contraste": [0.90, 1.10]},
        "fotometria_extra_finetune": {"brillo": [0.95, 1.05], "contraste": [0.95, 1.05]},
        "seleccion_checkpoint": "min_val_loss en validation_original",
        "sha256_manifiesto_260423": hash_archivo(MANIFIESTO_RUTA),
        "sha256_dataset": auditoria_actual["sha256_dataset"],
        "sha256_manifest_original": auditoria_actual["sha256_manifiesto"],
        "exclusiones_originales": auditoria_actual["exclusiones"],
        "versiones": {
            paquete: importlib.metadata.version(paquete)
            for paquete in ("tensorflow", "numpy", "pandas", "Pillow", "scikit-learn")
        },
        "python": platform.python_version(),
    }
    firma = hashlib.sha256(json.dumps(configuracion, sort_keys=True).encode()).hexdigest()
    configuracion["firma_protocolo"] = firma

    destino.mkdir(parents=True, exist_ok=False)
    (destino / "configuracion.json").write_text(
        json.dumps(configuracion, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (destino / "auditoria.json").write_text(
        json.dumps(auditoria_actual, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    manifiesto.to_csv(destino / "manifiesto_260423_usado.csv", index=False)
    plan_d1.to_csv(destino / "copias_russet_train.csv", index=False)
    train_efectivo.to_csv(destino / "manifiesto_train_efectivo.csv", index=False)
    estado_ruta = destino / "estado.json"
    estado_ruta.write_text(json.dumps({"estado": "iniciado"}), encoding="utf-8")
    inicio = time.perf_counter()

    try:
        # Etapa 1: entrenar la cabeza desde cero (sin pesos de s42).
        historial_cabeza = modelo.fit(
            entrenamiento_cabeza,
            steps_per_epoch=len(train_efectivo),
            validation_data=validacion,
            epochs=8,
            class_weight=pesos_efectivos,
            callbacks=callbacks_etapa(tf, destino, "cabeza"),
            verbose=1,
        )
        checkpoint_cabeza = destino / "mejor_cabeza.weights.h5"
        if not checkpoint_cabeza.is_file():
            raise RuntimeError("Falta el checkpoint de la cabeza.")
        modelo.load_weights(str(checkpoint_cabeza))

        # Etapa 2: la única diferencia es el alcance de conv5.
        base.trainable = True
        prefijo = "conv5_" if modo == "conv5" else "conv5_block3_"
        for capa in base.layers:
            capa.trainable = capa.name.startswith(prefijo) and not isinstance(
                capa, tf.keras.layers.BatchNormalization
            )
        capas_ajustables = [c.name for c in base.layers if c.trainable]
        if not capas_ajustables or any(not n.startswith(prefijo) for n in capas_ajustables):
            raise RuntimeError("El conjunto de capas entrenables no coincide.")
        if any(c.trainable for c in base.layers if isinstance(c, tf.keras.layers.BatchNormalization)):
            raise RuntimeError("BatchNormalization debe permanecer congelada.")
        n_parametros = sum(int(np.prod(v.shape)) for v in base.trainable_weights)
        (destino / "capas_finetune.json").write_text(
            json.dumps({"modo": modo, "n_capas": len(capas_ajustables),
                        "capas": capas_ajustables, "parametros_ajustables_backbone": n_parametros},
                       ensure_ascii=False, indent=2), encoding="utf-8"
        )
        print(f"Fine-tuning {modo}: {len(capas_ajustables)} capas, {n_parametros} parámetros.")

        modelo.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
            loss="sparse_categorical_crossentropy", metrics=["accuracy"],
        )
        historial_ft = modelo.fit(
            entrenamiento_ft,
            steps_per_epoch=len(train_efectivo),
            validation_data=validacion,
            epochs=8,
            class_weight=pesos_efectivos,
            callbacks=callbacks_etapa(tf, destino, "finetune"),
            verbose=1,
        )
        checkpoint_ft = destino / "mejor_finetune.weights.h5"
        if not checkpoint_ft.is_file():
            raise RuntimeError("Falta el checkpoint de fine-tuning.")
        modelo.load_weights(str(checkpoint_ft))

        met_val = evaluar_guardar(
            modelo, CONJUNTOS["validation_original"], "validation_original", destino, firma
        )
        for etapa, hist in (("cabeza", historial_cabeza), ("finetune", historial_ft)):
            fig, ax = plt.subplots(figsize=(7, 4))
            ax.plot(hist.history["loss"], label="train")
            ax.plot(hist.history["val_loss"], label="validation")
            ax.set(xlabel="Época", ylabel="Loss", title=f"{nombre}: {etapa}")
            ax.legend()
            fig.tight_layout()
            fig.savefig(destino / f"curva_loss_{etapa}.png", dpi=150)
            plt.close(fig)

        estado_ruta.write_text(json.dumps({
            "estado": "completo",
            "modo": modo,
            "semilla": semilla,
            "epocas_cabeza": len(historial_cabeza.history["loss"]),
            "mejor_epoca_cabeza": int(np.argmin(historial_cabeza.history["val_loss"])) + 1,
            "epocas_finetune": len(historial_ft.history["loss"]),
            "mejor_epoca_finetune": int(np.argmin(historial_ft.history["val_loss"])) + 1,
            "segundos_transcurridos": round(time.perf_counter() - inicio, 2),
            "evaluacion_exploratoria_consultada": False,
            "test_consultado": False,
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        (destino / "resumen_resultados.md").write_text(
            f"# Sprint 12 - {nombre}\n\n"
            f"- Validación (127): Macro-F1={met_val['macro_f1']:.4f}\n"
            f"- Precisión RUSSET={met_val['precision_russet']:.4f}\n"
            f"- Recall RUSSET={met_val['recall_russet']:.4f}\n"
            "- No se calcularon predicciones ni métricas de 260423 ni TEST.\n",
            encoding="utf-8"
        )
        print("Corrida completa:", destino.relative_to(RAIZ))
        print("No se evaluaron 260423 ni TEST.")
        return met_val
    except (Exception, KeyboardInterrupt) as error:
        import traceback
        estado_ruta.write_text(json.dumps({
            "estado": "incompleto", "detalle": str(error),
            "traceback": traceback.format_exc(),
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        raise
    finally:
        plt.close("all")
        tf.keras.backend.clear_session()


## 5. Ejecutar una corrida de manera controlada

In [ ]:
# Ejecutar solo UNA corrida por vez. Modifique estos 3 valores cuando corresponda.
EJECUTAR_EXPERIMENTO = False
MODO_ACTUAL = "conv5_block3"       # "conv5" o "conv5_block3"
SEMILLA_ACTUAL = 2026       # 123 o 2026 (42 ya está completada)

if EJECUTAR_EXPERIMENTO:
    ejecutar_una_corrida(MODO_ACTUAL, SEMILLA_ACTUAL)
    print("Cambie EJECUTAR_EXPERIMENTO a False y guarde. No repita la corrida.")
else:
    print("Modo seguro: no se entrena.")
    print("Siguiente corrida configurada:", MODO_ACTUAL, "semilla", SEMILLA_ACTUAL)
    print("El resumen se consulta en la sección 6.")


Integridad: 100/829
Integridad: 200/829
Integridad: 300/829
Integridad: 400/829
Integridad: 500/829
Integridad: 600/829
Integridad: 700/829
Integridad: 800/829
Integridad: 829/829
Epoch 1/8


/home/tci_auto/.venvs/palta_gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


584/584 ━━━━━━━━━━━━━━━━━━━━ 91s 149ms/step - accuracy: 0.6798 - loss: 0.7118 - val_accuracy: 0.6142 - val_loss: 0.7178 - learning_rate: 0.0010
Epoch 2/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 79s 136ms/step - accuracy: 0.8065 - loss: 0.4638 - val_accuracy: 0.4016 - val_loss: 1.5018 - learning_rate: 0.0010
Epoch 3/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 90s 154ms/step - accuracy: 0.8014 - loss: 0.4627 - val_accuracy: 0.8268 - val_loss: 0.3818 - learning_rate: 0.0010
Epoch 4/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 76s 130ms/step - accuracy: 0.8373 - loss: 0.3868 - val_accuracy: 0.8819 - val_loss: 0.2761 - learning_rate: 0.0010
Epoch 5/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 69s 117ms/step - accuracy: 0.8647 - loss: 0.3274 - val_accuracy: 0.8740 - val_loss: 0.3555 - learning_rate: 0.0010
Epoch 6/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 64s 110ms/step - accuracy: 0.8904 - loss: 0.2963 - val_accuracy: 0.8740 - val_loss: 0.2536 - learning_rate: 0.0010
Epoch 7/8
584/584 ━━━━━━━━━━━━━━━━━━━━ 60s 103ms/step - accuracy: 0.8596 - loss: 0.341

## 6. Comparación y estabilidad entre tres semillas

In [ ]:
# Solo lectura de métricas guardadas; no importa TensorFlow ni lee los TIFF.
REFERENCIAS_42 = {
    "conv5": RAIZ / "7. resultados/validacion_por_fecha/RN50_FECHA260423_s42_01",
    "conv5_block3": RAIZ / "7. resultados/finetuning_parcial/RN50_FT_CONV5B3_s42_01",
}
COLUMNAS_METRICAS = [
    "accuracy", "macro_f1", "precision_russet", "recall_russet",
    "average_precision_russet", "brier_russet",
]

filas = []
pendientes = []
firma_manifiesto = hash_archivo(MANIFIESTO_RUTA)
for semilla in (42, 123, 2026):
    for modo in MODOS:
        etiqueta = "CONV5" if modo == "conv5" else "CONV5B3"
        carpeta = (REFERENCIAS_42[modo] if semilla == 42 else
                   SALIDA / f"RN50_{etiqueta}_s{semilla}_01")
        ruta_metricas = carpeta / "metricas_validation_original.json"
        ruta_estado = carpeta / "estado.json"
        if not ruta_metricas.is_file() or not ruta_estado.is_file():
            pendientes.append(f"{modo} / semilla {semilla}")
            continue
        estado = json.loads(ruta_estado.read_text(encoding="utf-8"))
        if estado.get("estado") != "completo":
            pendientes.append(f"{modo} / semilla {semilla} (incompleta)")
            continue
        configuracion = json.loads((carpeta / "configuracion.json").read_text(encoding="utf-8"))
        if configuracion.get("sha256_manifiesto_260423") != firma_manifiesto:
            raise ValueError(f"El manifiesto no coincide con el histórico: {carpeta}")
        met = json.loads(ruta_metricas.read_text(encoding="utf-8"))
        if met.get("particion") != "validation_original" or met.get("n_imagenes") != 127:
            raise ValueError(f"Evaluación no comparable: {ruta_metricas}")
        fila = {"modo": modo, "semilla": semilla, "n": met["n_imagenes"],
                "origen": "histórico" if semilla == 42 else "sprint12"}
        fila.update({c: met[c] for c in COLUMNAS_METRICAS})
        filas.append(fila)

if filas:
    tabla = pd.DataFrame(filas).sort_values(["semilla", "modo"])
    display(tabla.round(4))
else:
    print("Todavía no hay métricas disponibles.")

if pendientes:
    print("Corridas pendientes:", ", ".join(pendientes))
    print("Aún no corresponde calcular media, desviación ni declarar un mejor método.")
else:
    tabla = tabla.sort_values(["modo", "semilla"])
    resumen = (tabla.groupby("modo")[COLUMNAS_METRICAS]
               .agg(["mean", "std"]).round(4))
    display(resumen)
    pares = tabla.pivot(index="semilla", columns="modo", values=COLUMNAS_METRICAS)
    diferencias = pd.DataFrame({
        c: pares[(c, "conv5_block3")] - pares[(c, "conv5")]
        for c in COLUMNAS_METRICAS
    })
    print("Diferencia por semilla: conv5_block3 menos conv5")
    display(diferencias.round(4))
    print("Diferencia media y desviación entre semillas (descriptivo):")
    display(diferencias.agg(["mean", "std"]).round(4))
    SALIDA.mkdir(parents=True, exist_ok=True)
    tabla.to_csv(SALIDA / "comparacion_tres_semillas.csv", index=False)
    resumen.to_csv(SALIDA / "resumen_media_desviacion.csv")
    diferencias.to_csv(SALIDA / "diferencias_pareadas_por_semilla.csv")
    print("Archivos comparativos guardados en", SALIDA.relative_to(RAIZ))
    print("Tres semillas no prueban generalización ni significancia estadística.")
    print("No se calcularon predicciones ni métricas de 260423 o TEST.")


## Interpretación

Las diferencias entre métodos se evalúan con métricas de la misma partición y una comparación emparejada por semilla. Cada semilla cambia la inicialización, el orden de entrenamiento y las transformaciones aleatorias; ambos métodos deben usar la misma semilla para preservar el emparejamiento.

Media y desviación con solo tres semillas sirven para examinar estabilidad, pero no constituyen prueba de superioridad estadística. La evidencia de generalización necesita un conjunto externo independiente. No se modifica el umbral de decisión, ni el número de épocas, ni los hiperparámetros, ni la separación de datos.